In [0]:
orders_df = spark.read.csv('/Volumes/olist_ecommerce/default/raw_data/orders.csv',header = True)
orders_df.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

Splitting data into three:\
initial_load : ~60%\
increment_1: ~20%\
increment_2: ~20%

In [0]:
from pyspark.sql.functions import col

orders_init = orders_df.filter(
    col('order_purchase_timestamp') < '2017-12-01'
)
orders_inc1 = orders_df.filter(
    (col('order_purchase_timestamp') >= '2017-12-01') &
    (col('order_purchase_timestamp') < '2018-05-01')
)
orders_inc2 = orders_df.filter(
    col('order_purchase_timestamp') >= '2018-05-01'
)


Verifying if the split is right

In [0]:
print(orders_init.count()+orders_inc1.count()+orders_inc2.count())
print(orders_df.count())

99441
99441


In [0]:
init_order_ids = orders_init.select('order_id')
inc1_order_ids = orders_inc1.select('order_id')
inc2_order_ids = orders_inc2.select('order_id')

In [0]:
order_items_df = spark.read.csv('/Volumes/olist_ecommerce/default/raw_data/order_items.csv',header = True)
order_items_init = order_items_df.join(init_order_ids, "order_id", "inner")
order_items_inc1 = order_items_df.join(inc1_order_ids, "order_id", "inner")
order_items_inc2 = order_items_df.join(inc2_order_ids, "order_id", "inner")

payment_df = spark.read.csv('/Volumes/olist_ecommerce/default/raw_data/order_payments.csv',header = True)
payments_init = payment_df.join(init_order_ids, "order_id", "inner")
payments_inc1 = payment_df.join(inc1_order_ids, "order_id", "inner")
payments_inc2 = payment_df.join(inc2_order_ids, "order_id", "inner")

# approx 364 reviews excluded as their order_id has no match in orders.csv, excluded intentionally as they can be linked to products, customer etc.
order_review_df = spark.read.csv('/Volumes/olist_ecommerce/default/raw_data/order_reviews.csv',header = True, quote='"', escape='"', multiLine=True)
reviews_init = order_review_df.join(init_order_ids, "order_id", "inner")
reviews_inc1 = order_review_df.join(inc1_order_ids, "order_id", "inner")
reviews_inc2 = order_review_df.join(inc2_order_ids, "order_id", "inner")


In [0]:
def write_single_csv(df, path):
    df.coalesce(1).write.csv(path + '_tmp', header=True, mode='overwrite')
    part_file = [f.path for f in dbutils.fs.ls(path + '_tmp') 
                 if f.name.startswith('part-')][0]
    dbutils.fs.cp(part_file, path)
    dbutils.fs.rm(path + '_tmp', recurse=True)
    print(f'{path}')

base = '/Volumes/olist_ecommerce/default/incremental_loading/'

datasets = {
    'initial/orders.csv': orders_init,
    'initial/order_items.csv': order_items_init,
    'initial/order_payments.csv': payments_init,
    'initial/order_reviews.csv': reviews_init,

    'increment1/orders.csv': orders_inc1,
    'increment1/order_items.csv': order_items_inc1,
    'increment1/order_payments.csv': payments_inc1,
    'increment1/order_reviews.csv': reviews_inc1,

    'increment2/orders.csv': orders_inc2,
    'increment2/order_items.csv': order_items_inc2,
    'increment2/order_payments.csv': payments_inc2,
    'increment2/order_reviews.csv': reviews_inc2
}

for file_path, dataframe in datasets.items():
    write_single_csv(dataframe, base + file_path)

/Volumes/olist_ecommerce/default/incremental_loading/initial/orders.csv
/Volumes/olist_ecommerce/default/incremental_loading/initial/order_items.csv
/Volumes/olist_ecommerce/default/incremental_loading/initial/order_payments.csv
/Volumes/olist_ecommerce/default/incremental_loading/initial/order_reviews.csv
/Volumes/olist_ecommerce/default/incremental_loading/increment1/orders.csv
/Volumes/olist_ecommerce/default/incremental_loading/increment1/order_items.csv
/Volumes/olist_ecommerce/default/incremental_loading/increment1/order_payments.csv
/Volumes/olist_ecommerce/default/incremental_loading/increment1/order_reviews.csv
/Volumes/olist_ecommerce/default/incremental_loading/increment2/orders.csv
/Volumes/olist_ecommerce/default/incremental_loading/increment2/order_items.csv
/Volumes/olist_ecommerce/default/incremental_loading/increment2/order_payments.csv
/Volumes/olist_ecommerce/default/incremental_loading/increment2/order_reviews.csv
